# Single Task Visualization for MAML/MLP Models

This notebook visualizes model performance on a single random task.

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import OrderedDict

# Import utility functions
from data_management_utils import (
    analyze_continuity,
    load_and_normalize_data,
    apply_normalization
)

# MAML import
sys.path.append('../../model_code/')
from maml_optimized import OptimizedMAML, MAMLModel_3hidden
from networks import MLP_Aadam, MLP

In [ ]:
# Configuration
PDK = 'ASAP7'  # 'ASAP7' or 'TSMC'
TEST_TYPE = 'topology_agnostic'  # 'topology_agnostic' or 'intra_topology'
DATA_TYPE = 'transition'  # 'cell' or 'transition'
MODEL_TYPE = 'MAML'  # 'MAML' or 'MLP' or 'Aadam'
MODE = 'extrapolation'  # 'extrapolation' or 'interpolation'

# MAML specific (only used if MODEL_TYPE == 'MAML')
INNER = 1
INNERDIV = 100
META = 32

# MLP specific (only used if MODEL_TYPE in ['MLP', 'Aadam'])
NUM_ITERATIONS = 300000  # pretraining iterations

GPU_ID = '0'
CELL_NAME = 'FAx1'  # For testing specific cell

# Sampling indices based on mode
if MODE == 'extrapolation':
    INDICES = [5, 30, 55]
    LEFT_BOUND = 5
    RIGHT_BOUND = 56
else:  # interpolation
    INDICES = [0, 15, 30, 45, 60]  # Including endpoints
    LEFT_BOUND = 0
    RIGHT_BOUND = 61

RANDOM_TASK_ID = None  # Set to None for random selection, or specify a number

In [ ]:
# GPU settings
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Device:', device)
if torch.cuda.is_available():
    print('Current cuda device:', torch.cuda.current_device())
    print('Count of using GPUs:', torch.cuda.device_count())

In [ ]:
# Load training data for normalization statistics
print("Loading training data for normalization...")

if PDK == 'ASAP7':
    if TEST_TYPE == 'topology_agnostic':
        train_data_paths = [
            (f"/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/unified_invbuf/merged_invbuf_input_{DATA_TYPE}.pth",
             f"/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/unified_invbuf/merged_invbuf_output_{DATA_TYPE}.pth"),
            (f"/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_ASAP7/topology_agnostic_data_upgraded/{DATA_TYPE}_topology_agnostic_train_input.pth",
             f"/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_ASAP7/topology_agnostic_data_upgraded/{DATA_TYPE}_topology_agnostic_train_output.pth")
        ]
    else:  # intra_topology
        train_data_paths = [
            (f"/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_ASAP7/intra_topology_data_upgraded/{DATA_TYPE}_intratopology_train_input.pth",
             f"/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_ASAP7/intra_topology_data_upgraded/{DATA_TYPE}_intratopology_train_output.pth"),
            (f"/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/unified_invbuf/merged_invbuf_input_{DATA_TYPE}.pth",
             f"/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/unified_invbuf/merged_invbuf_output_{DATA_TYPE}.pth")
        ]
else:  # TSMC
    if TEST_TYPE == 'topology_agnostic':
        train_data_paths = [
            (f"/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_tsmc_processed/topology_agnostic_data/tsmc_topology_agnostic_train_input_{DATA_TYPE}.pth",
             f"/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_tsmc_processed/topology_agnostic_data/tsmc_topology_agnostic_train_output_{DATA_TYPE}.pth")
        ]
    else:  # intra_topology
        train_data_paths = [
            (f"/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_tsmc_processed/intra_topology_data/tsmc_intra_topology_train_input_{DATA_TYPE}.pth",
             f"/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_tsmc_processed/intra_topology_data/tsmc_intra_topology_train_output_{DATA_TYPE}.pth")
        ]

norm_stats = load_and_normalize_data(train_data_paths)
print("✅ Normalization statistics calculated")

In [ ]:
# Load model
print(f"\nLoading {MODEL_TYPE} model...")

input_features = 9

if MODEL_TYPE == 'MAML':
    maml_model = OptimizedMAML(
        model=MAMLModel_3hidden(in_features=input_features, layer_length=40),
        dataset_in=None,
        dataset_out=None,
        inner_lr=0.001,
        meta_lr=0.0001
    )
    
    if PDK == 'ASAP7':
        if TEST_TYPE == 'topology_agnostic':
            model_path = f"/home/tkdgn2907/Deepsets_test/MAML/Projects/pretrained_models/checkpoints/taskdivide_all_checkpoints/{DATA_TYPE}_innerdiv{INNERDIV}_meta{META}_topology_agnostic_519traintask_full1DMAML_weights_3hidden_(40)_600000_inner{INNER}_upgraded.pth"
        else:
            model_path = f"/home/tkdgn2907/Deepsets_test/MAML/Projects/pretrained_models/checkpoints/taskdivide_all_checkpoints/{DATA_TYPE}_innerdiv{INNERDIV}_meta{META}_intratopology_519traintask_full1DMAML_weights_3hidden_(40)_300000_inner{INNER}_upgraded.pth"
    else:  # TSMC
        if TEST_TYPE == 'topology_agnostic':
            model_path = f"/home/tkdgn2907/Deepsets_test/MAML/Projects/pretrained_models/taskdivide_all/{DATA_TYPE}_innerdiv{INNERDIV}_meta{META}_topology_agnostic_519traintask_full1DMAML_weights_3hidden_(40)_300000_inner{INNER}_upgraded_tsmc.pth"
        else:
            model_path = f"/home/tkdgn2907/Deepsets_test/MAML/Projects/pretrained_models/taskdivide_all/{DATA_TYPE}_innerdiv{INNERDIV}_meta{META}_intra_topology_519traintask_full1DMAML_weights_3hidden_(40)_300000_inner{INNER}_upgraded_tsmc_merged.pth"
    
    state_dict = torch.load(model_path, map_location=device)
    maml_model.model.load_state_dict(state_dict)
    model = maml_model.model.model
    
elif MODEL_TYPE == 'Aadam':
    model = MLP_Aadam(input_size=input_features, output_size=1).to(device)
    
    if PDK == 'ASAP7':
        if TEST_TYPE == 'topology_agnostic':
            model_path = f"../model_pretraining_code/MLP_pretrained_model/pretrained_mlp1_topology_agnostic_{DATA_TYPE}_aadam_{NUM_ITERATIONS}.pth"
        else:
            model_path = f"/home/tkdgn2907/Deepsets_test/MAML/Projects/pretraining/model_pretraining_code/MLP_pretrained_model/pretrained_mlp1_intratopology_{DATA_TYPE}_aadam_{NUM_ITERATIONS}.pth"
    else:  # TSMC
        if TEST_TYPE == 'topology_agnostic':
            model_path = f"../model_pretraining_code/MLP_pretrained_model/pretrained_mlp1_topology_agnostic_{DATA_TYPE}_tsmc_aadam_{NUM_ITERATIONS}.pth"
        else:
            model_path = f"../model_pretraining_code/MLP_pretrained_model/pretrained_mlp1_intratopology_{DATA_TYPE}_tsmc_aadam_{NUM_ITERATIONS}.pth"
    
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
else:  # MLP
    model = MLP(input_size=input_features, output_size=1).to(device)
    
    if PDK == 'ASAP7':
        if TEST_TYPE == 'topology_agnostic':
            model_path = f"../model_pretraining_code/MLP_pretrained_model/pretrained_mlp1_topology_agnostic_{DATA_TYPE}_mlp_{NUM_ITERATIONS}.pth"
        else:
            model_path = f"/home/tkdgn2907/Deepsets_test/MAML/Projects/pretraining/model_pretraining_code/MLP_pretrained_model/pretrained_mlp1_intratopology_{DATA_TYPE}_mlp_{NUM_ITERATIONS}.pth"
    else:  # TSMC
        if TEST_TYPE == 'topology_agnostic':
            model_path = f"../model_pretraining_code/MLP_pretrained_model/pretrained_mlp1_topology_agnostic_{DATA_TYPE}_tsmc_mlp_{NUM_ITERATIONS}.pth"
        else:
            model_path = f"../model_pretraining_code/MLP_pretrained_model/pretrained_mlp1_intratopology_{DATA_TYPE}_tsmc_mlp_{NUM_ITERATIONS}.pth"
    
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])

print(f"✅ Loaded model: {model_path}")

In [ ]:
# Load test data
print("\nLoading test data...")

if PDK == 'ASAP7':
    if TEST_TYPE == 'topology_agnostic':
        data_dir = "/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_ASAP7/test_topology_agnostic"
        test_input_path = f"{data_dir}/{CELL_NAME}/{DATA_TYPE}_{CELL_NAME}_test_input.pth"
        test_output_path = f"{data_dir}/{CELL_NAME}/{DATA_TYPE}_{CELL_NAME}_test_output.pth"
    else:  # intra_topology
        data_dir = "/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_ASAP7/intra_topology_data_upgraded"
        test_input_path = f"{data_dir}/{CELL_NAME}/{DATA_TYPE}_{CELL_NAME}_test_input.pth"
        test_output_path = f"{data_dir}/{CELL_NAME}/{DATA_TYPE}_{CELL_NAME}_test_output.pth"
else:  # TSMC
    if TEST_TYPE == 'topology_agnostic':
        data_dir = "/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_tsmc_processed/topology_agnostic_data"
        test_input_path = f"{data_dir}/{CELL_NAME}/tsmc_merged_test_input_{DATA_TYPE}.pth"
        test_output_path = f"{data_dir}/{CELL_NAME}/tsmc_merged_test_output_{DATA_TYPE}.pth"
    else:  # intra_topology
        data_dir = "/home/tkdgn2907/Deepsets_test/MAML/Projects/dataset_all/dataset_tsmc_processed/intra_topology_data"
        test_input_path = f"{data_dir}/{CELL_NAME}/tsmc_merged_test_input_{DATA_TYPE}.pth"
        test_output_path = f"{data_dir}/{CELL_NAME}/tsmc_merged_test_output_{DATA_TYPE}.pth"

test_data_input = torch.load(test_input_path)
test_data_output = torch.load(test_output_path)

# Add dimension to output if needed
if len(test_data_output.shape) == 2:
    test_data_output = test_data_output.unsqueeze(-1)

print(f"Test input shape: {test_data_input.shape}")
print(f"Test output shape: {test_data_output.shape}")

# Apply normalization
apply_normalization(test_data_input, norm_stats)

# Move to GPU
test_data_input = test_data_input.to(device)
test_data_output = test_data_output.to(device)

print("✅ Test data loaded and normalized")

In [ ]:
def model_functions_at_training(initial_model, X, y, sampled_steps, x_axis, true_x, true_function, 
                               optim=torch.optim.SGD, lr=0.003, adam_step=0, std=1, mean=10,
                               left_bound=5, right_bound=56, total_points=61, mode='extrapolation'):
    """
    Trains the model on X, y and measures the loss curve.
    For each n in sampled_steps, records model(x_axis) after n gradient updates.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    X = X.to(device)
    y = y.to(device)
    true_x = true_x.to(device)
    true_function = true_function.to(device)
    std = std.to(device) if isinstance(std, torch.Tensor) else torch.tensor(std).to(device)
    mean = mean.to(device) if isinstance(mean, torch.Tensor) else torch.tensor(mean).to(device)
    
    # Create x_axis_tensor and ensure it has same features as X
    x_axis_tensor = torch.tensor(x_axis, dtype=torch.float).view(-1, 1).to(device)
    # Add remaining features (using the same values as X[0])
    feature_vector_before = X[0, :4]  # [a_param, b_param, c_param, temperature]
    feature_vector_after = X[0, 5:]   # [additional_dim, delay_indicator, slew, load_cap]

    # Combine: [0:4] + [voltage] + [5:8]
    x_axis_tensor_extended = torch.cat([
        feature_vector_before.unsqueeze(0).repeat(len(x_axis), 1),  # [1000, 4]
        x_axis_tensor,                                               # [1000, 1]
        feature_vector_after.unsqueeze(0).repeat(len(x_axis), 1)    # [1000, 4]
    ], dim=1)  # Final: [1000, 9]
    
    # Copy model into a new object to preserve original weights during training
    input_dim = X.shape[2] if len(X.shape) > 2 else X.shape[1]
    
    if MODEL_TYPE == 'Aadam':
        new_model = MLP_Aadam(input_size=input_dim, output_size=1).to(device)
    elif MODEL_TYPE == 'MLP':
        new_model = MLP(input_size=input_dim, output_size=1).to(device)
    else:  # MAML or default
        new_model = nn.Sequential(OrderedDict([
            ('l1', nn.Linear(input_dim, 40)),
            ('relu1', nn.ReLU()),
            ('l2', nn.Linear(40, 40)),
            ('relu3', nn.ReLU()),
            ('l4', nn.Linear(40, 40)),
            ('relu2', nn.ReLU()),
            ('l3', nn.Linear(40, 1))
        ])).to(device)
    
    new_model.load_state_dict(initial_model.state_dict())
    criterion = nn.MSELoss()
    optimiser = optim(new_model.parameters(), lr, weight_decay=1e-4)
    adam_condition_triggered = False
    
    # Train model on a random task
    num_steps = max(sampled_steps)
    K = X.shape[0]
    
    losses = []
    outputs = {}
    
    # Record initial output
    with torch.no_grad():
        outputs['initial'] = new_model(x_axis_tensor_extended).cpu().numpy().flatten()
    
    loss = criterion(new_model(X), y) / K
    
    # Adam training if initial loss is high
    if loss > 1e-4:
        adam_condition_triggered = True
        optimiser2 = torch.optim.Adam(new_model.parameters(), lr=3e-4, weight_decay=1e-4)        
        for step in range(1, adam_step + 1):
            loss = criterion(new_model(X), y) / K
            losses.append(loss.item())

            # Compute grad and update inner loop weights
            new_model.zero_grad()
            loss.backward()
            optimiser2.step()
            
            # Record outputs at sampled steps
            if step in sampled_steps:
                with torch.no_grad():
                    outputs[step] = new_model(x_axis_tensor_extended).cpu().numpy().flatten()

    # Calculate losses for different regions
    total_loss = 0
    total_inter_loss = 0
    total_rightex_loss = 0
    total_leftex_loss = 0
    total_mape_loss = 0
    total_leftex_mape = 0
    total_inter_mape = 0
    total_rightex_mape = 0
    total_mae = 0
    total_leftex_mae = 0
    total_inter_mae = 0
    total_rightex_mae = 0
    
    # Store predictions and actual values for plotting
    predictions = []
    actual_values = []
    
    for i in range(total_points):
        pred_value = ((new_model(true_x[i])) * std + mean).item()
        actual_value = ((true_function[i]) * std + mean).item()
        
        predictions.append(pred_value)
        actual_values.append(actual_value)
        
        loss = criterion((new_model(true_x[i])) * std + mean, (true_function[i]) * std + mean)
        
        # Calculate MAPE (Mean Absolute Percentage Error)
        if abs(actual_value) > 1e-8:  # Avoid division by zero
            mape_loss = abs((pred_value - actual_value) / actual_value)
        else:
            mape_loss = 0
        
        # Calculate MAE
        mae = abs(pred_value - actual_value)
        
        total_loss += loss
        total_mape_loss += mape_loss
        total_mae += mae
        
        # Regional calculations based on mode
        if mode == 'extrapolation':
            if i < left_bound:  # Left extrapolation region
                total_leftex_loss += loss
                total_leftex_mape += mape_loss
                total_leftex_mae += mae
            elif i < right_bound:  # Interpolation region
                total_inter_loss += loss
                total_inter_mape += mape_loss
                total_inter_mae += mae
            else:  # Right extrapolation region
                total_rightex_loss += loss
                total_rightex_mape += mape_loss
                total_rightex_mae += mae
        else:  # interpolation mode - all points are in interpolation region
            if i < left_bound or i >= right_bound:
                continue  # Skip endpoints in some calculations
            total_inter_loss += loss
            total_inter_mape += mape_loss
            total_inter_mae += mae
    
    # Calculate average losses and MAPEs for each region
    avg_total_loss = total_loss / total_points
    
    if mode == 'extrapolation':
        avg_inter_loss = total_inter_loss / (right_bound - left_bound) if (right_bound - left_bound) > 0 else 0
        avg_rightex_loss = total_rightex_loss / (total_points - right_bound) if (total_points - right_bound) > 0 else 0
        avg_leftex_loss = total_leftex_loss / left_bound if left_bound > 0 else 0
        avg_total_mape = total_mape_loss / total_points
        avg_leftex_mape = total_leftex_mape / left_bound if left_bound > 0 else 0
        avg_inter_mape = total_inter_mape / (right_bound - left_bound) if (right_bound - left_bound) > 0 else 0
        avg_rightex_mape = total_rightex_mape / (total_points - right_bound) if (total_points - right_bound) > 0 else 0
        avg_total_mae = total_mae / total_points
        avg_leftex_mae = total_leftex_mae / left_bound if left_bound > 0 else 0
        avg_inter_mae = total_inter_mae / (right_bound - left_bound) if (right_bound - left_bound) > 0 else 0
        avg_rightex_mae = total_rightex_mae / (total_points - right_bound) if (total_points - right_bound) > 0 else 0
    else:  # interpolation
        avg_inter_loss = total_inter_loss / (right_bound - left_bound) if (right_bound - left_bound) > 0 else 0
        avg_rightex_loss = 0
        avg_leftex_loss = 0
        avg_total_mape = total_mape_loss / total_points
        avg_leftex_mape = 0
        avg_inter_mape = total_inter_mape / (right_bound - left_bound) if (right_bound - left_bound) > 0 else 0
        avg_rightex_mape = 0
        avg_total_mae = total_mae / total_points
        avg_leftex_mae = 0
        avg_inter_mae = total_inter_mae / (right_bound - left_bound) if (right_bound - left_bound) > 0 else 0
        avg_rightex_mae = 0
    
    return (new_model, outputs, losses, avg_total_loss, avg_inter_loss, avg_rightex_loss, avg_leftex_loss, 
            avg_total_mape, avg_leftex_mape, avg_inter_mape, avg_rightex_mape, predictions, actual_values, 
            adam_condition_triggered, avg_total_mae, avg_leftex_mae, avg_inter_mae, avg_rightex_mae)

In [ ]:
def plot_sampled_performance(initial_model, model_name, X, y, true_x, true_function, grad, move, 
                            optim=torch.optim.SGD, lr=0.001,
                            left_bound=5, right_bound=56, total_points=61, mode='extrapolation'):
    """Plots model performance on a single task."""
    x_axis = np.linspace(-1.7, 1.7, 1000)
    sampled_steps = [30]
    
    y_mean = y.mean() 
    y_std = y.std() 
    mean_values = [y_mean]
    std_values = [y_std * grad]
    loss_min = 10000 
    inter_loss_min = 10000
    rightex_loss_min = 10000
    leftex_loss_min = 10000
    mape_min = 10000
    
    # Store all predictions and actuals for final plotting
    all_predictions = []
    all_actuals = []
    
    # For each combination of mean and std
    for mean in mean_values:
        for std in std_values:
            y_mean1 = mean  # Update mean
            y_std1 = std    # Update std
            
            y_test = (y - y_mean1) / y_std1 + move
            true_function1 = (true_function - y_mean1) / y_std1 + move
            
            # Pass the updated mean and std to model_functions_at_training
            (trained_model, outputs, losses, total_loss, total_inter_loss, total_rightex_loss, total_leftex_loss, 
             total_mape_loss, leftex_mape, inter_mape, rightex_mape, predictions, actual_values, adam_condition_triggered,
             total_mae, leftex_mae, inter_mae, rightex_mae) = model_functions_at_training(
                initial_model,
                X, y=y_test,
                sampled_steps=sampled_steps,
                x_axis=x_axis,
                true_x=true_x,
                true_function=true_function1,
                optim=optim, 
                lr=lr,
                adam_step=40,
                std=y_std1,
                mean=y_mean1,
                left_bound=left_bound,
                right_bound=right_bound,
                total_points=total_points,
                mode=mode
            )
            adam_used = adam_condition_triggered
            
            # Collect predictions and actuals
            all_predictions.extend(predictions)
            all_actuals.extend(actual_values)
            
            model_min = trained_model
            loss_min = total_loss
            mean_min = mean
            std_min = std
            output_min = outputs
            losses_min = losses
            inter_loss_min = total_inter_loss
            leftex_loss_min = total_leftex_loss
            rightex_loss_min = total_rightex_loss
            mape_min = total_mape_loss
            leftex_mape_min = leftex_mape
            inter_mape_min = inter_mape
            rightex_mape_min = rightex_mape
            mae_min = total_mae
            leftex_mae_min = leftex_mae
            inter_mae_min = inter_mae
            rightex_mae_min = rightex_mae
    
    # Move tensors to CPU for plotting
    model_min = model_min.to("cpu")
    true_x = true_x.cpu()
    true_function = true_function.cpu()
    mean_min = mean_min.clone().detach().cpu() if isinstance(mean_min, torch.Tensor) else mean_min
    std_min = std_min.clone().detach().cpu() if isinstance(std_min, torch.Tensor) else std_min
    X = X.clone().detach().cpu()
    y = y.clone().detach().cpu()
    move = move.cpu() if isinstance(move, torch.Tensor) else move
    
    # Extract original feature (voltage, index 4) for plotting
    true_x_plot = true_x[:, 4] if true_x.dim() > 1 and true_x.shape[1] > 1 else true_x.squeeze()
    X_plot = X[:, 4] if X.dim() > 1 and X.shape[1] > 1 else X.squeeze()
    
    plt.figure(figsize=(15, 5))
    
    # Plot model functions
    plt.subplot(1, 2, 1)
    plt.scatter(true_x_plot, (true_function - mean_min) / std_min + move, label='All Data Points', color='orange', alpha=0.5)
    plt.scatter(X_plot, (y - mean_min) / std_min + move, label='Support Set', color='blue', s=100, zorder=5)
    
    if 'initial' in output_min:
        plt.plot(x_axis, output_min['initial'], ':', color=(0.7, 0, 0, 1), label='Initial weights', linewidth=2)
    
    for step in sampled_steps:
        if step in output_min:
            plt.plot(x_axis, output_min[step],
                    '-.' if step == 1 else '-', color=(0.5, 0, 0, 1),
                    label=f'After {step} steps', linewidth=2)

    plt.title(f"Model fit: {model_name} ({MODE} mode)", fontsize=14)
    plt.xlabel("Voltage (normalized)", fontsize=12)
    plt.ylabel("Delay (normalized)", fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Plot losses
    plt.subplot(1, 2, 2)
    if len(losses_min) > 5:
        plt.plot(losses_min[5:], linewidth=2)
    else:
        plt.plot(losses_min, linewidth=2)
    plt.title("Loss over time", fontsize=14)
    plt.xlabel("Gradient steps taken", fontsize=12)
    plt.ylabel("MSE Loss", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return (loss_min, inter_loss_min, leftex_loss_min, rightex_loss_min, 
            mape_min, leftex_mape_min, inter_mape_min, rightex_mape_min,
            all_predictions, all_actuals, output_min, model_min, mean_min, std_min, move, losses_min, adam_used,
            mae_min, leftex_mae_min, inter_mae_min, rightex_mae_min)

In [ ]:
# Select a random task or use specified task ID
if RANDOM_TASK_ID is None:
    randomtask = random.randint(0, len(test_data_input) - 1)
else:
    randomtask = RANDOM_TASK_ID

print(f"\n" + "="*80)
print(f"Selected task: {randomtask}")
print(f"PDK: {PDK}, Type: {TEST_TYPE}, Cell: {CELL_NAME}")
print(f"Model: {MODEL_TYPE}, Mode: {MODE}")
print(f"Support set indices: {INDICES}")
print("="*80)

K = len(INDICES)

# Sample data at specified indices
X = test_data_input[randomtask][INDICES]
y = test_data_output[randomtask][INDICES]

# Define regions
if MODE == 'extrapolation':
    testdata_rightex_output = test_data_output[randomtask][RIGHT_BOUND:]
    testdata_leftex_output = test_data_output[randomtask][:LEFT_BOUND]
    testdata_inter_output = test_data_output[randomtask][LEFT_BOUND:RIGHT_BOUND]
    
    y_leftex_mean = testdata_leftex_output.mean()
    y_rightex_mean = testdata_rightex_output.mean()
    y_inter_mean = testdata_inter_output.mean()
else:  # interpolation
    testdata_inter_output = test_data_output[randomtask][LEFT_BOUND:RIGHT_BOUND]
    y_inter_mean = testdata_inter_output.mean()
    y_leftex_mean = 0
    y_rightex_mean = 0

y1_mean = test_data_output[randomtask].mean()

y_mean = y.mean()
y_std = y.std()

print(f"\nTask statistics:")
print(f"  Support set mean: {y_mean.item():.6f}")
print(f"  Support set std: {y_std.item():.6f}")
print(f"  Full task mean: {y1_mean.item():.6f}")

if y_std > 1e-8 and y1_mean > 1e-8:
    y_norm = (y - y_mean) / y_std
    
    # Create center input for 9D
    center_input = torch.zeros((1, X.shape[1])).to(device)
    center_input[0, 4] = 0.0  # voltage = 0 (normalized)
    center_input[0, :4] = X[0, :4]  # copy first 4 features
    center_input[0, 5:] = X[0, 5:]  # copy remaining features
    center = model(center_input).item()
    
    y_max = y_norm[:, 0].max()
    y_min = y_norm[:, 0].min()
    
    # Get model predictions for scaling
    predictions = model(test_data_input[randomtask][LEFT_BOUND:RIGHT_BOUND])
    min_val = predictions.min().item()
    max_val = predictions.max().item()
    
    if abs(max_val - min_val) > 1e-8:
        grad = (y_max - y_min) / (max_val - min_val)
        middle_idx = len(INDICES) // 2
        move = center - y_norm[middle_idx, 0] / grad
        
        print(f"\nScaling parameters:")
        print(f"  Gradient: {grad:.6f}")
        print(f"  Move: {move:.6f}")
        print(f"  Center: {center:.6f}")
        
        # Run model and plot
        (total_loss1, inter_loss1, leftex_loss1, rightex_loss1, 
        mape_loss1, leftex_mape1, inter_mape1, rightex_mape1,
        predictions, actual_values, _, _, _, _, _, _, adam_used,
        mae_loss1, leftex_mae1, inter_mae1, rightex_mae1) = plot_sampled_performance(
            model, MODEL_TYPE, X, y,
            test_data_input[randomtask], test_data_output[randomtask], grad, move,
            left_bound=LEFT_BOUND, right_bound=RIGHT_BOUND, total_points=61, mode=MODE
        )
        
        # Calculate NRMSE
        nrmse1 = (total_loss1 ** 0.5) / (y1_mean + 1e-8) * 100
        nrmse_inter = (inter_loss1 ** 0.5) / (y_inter_mean + 1e-8) * 100
        
        print(f"\n" + "="*80)
        print("RESULTS")
        print("="*80)
        
        if MODE == 'extrapolation':
            nrmse_leftex = (leftex_loss1 ** 0.5) / (y_leftex_mean + 1e-8) * 100
            nrmse_rightex = (rightex_loss1 ** 0.5) / (y_rightex_mean + 1e-8) * 100
            mape_l_percent = leftex_mape1 * 100
            mape_r_percent = rightex_mape1 * 100
            
            print(f"\nNRMSE (Normalized Root Mean Square Error):")
            print(f"  Total:               {nrmse1.item():.3f}%")
            print(f"  Interpolation:       {nrmse_inter.item():.3f}%")
            print(f"  Left Extrapolation:  {nrmse_leftex.item():.3f}%")
            print(f"  Right Extrapolation: {nrmse_rightex.item():.3f}%")
            
            print(f"\nMAPE (Mean Absolute Percentage Error):")
            print(f"  Total:               {mape_loss1 * 100:.3f}%")
            print(f"  Interpolation:       {inter_mape1 * 100:.3f}%")
            print(f"  Left Extrapolation:  {mape_l_percent:.3f}%")
            print(f"  Right Extrapolation: {mape_r_percent:.3f}%")
            
            print(f"\nMAE (Mean Absolute Error):")
            print(f"  Total:               {mae_loss1 * 1000:.3f} ps")
            print(f"  Interpolation:       {inter_mae1 * 1000:.3f} ps")
            print(f"  Left Extrapolation:  {leftex_mae1 * 1000:.3f} ps")
            print(f"  Right Extrapolation: {rightex_mae1 * 1000:.3f} ps")
        else:  # interpolation
            print(f"\nNRMSE (Normalized Root Mean Square Error):")
            print(f"  Total:               {nrmse1.item():.3f}%")
            print(f"  Interpolation:       {nrmse_inter.item():.3f}%")
            
            print(f"\nMAPE (Mean Absolute Percentage Error):")
            print(f"  Total:               {mape_loss1 * 100:.3f}%")
            print(f"  Interpolation:       {inter_mape1 * 100:.3f}%")
            
            print(f"\nMAE (Mean Absolute Error):")
            print(f"  Total:               {mae_loss1 * 1000:.3f} ps")
            print(f"  Interpolation:       {inter_mae1 * 1000:.3f} ps")
        
        print(f"\nAdam condition triggered: {adam_used}")
        print("="*80)
    else:
        print("\n⚠️ Error: max_val == min_val, cannot compute gradient")
else:
    print("\n⚠️ Error: y_std or y1_mean too small")

In [ ]:
# Optional: Plot prediction vs actual scatter plot
if 'predictions' in locals() and 'actual_values' in locals():
    plt.figure(figsize=(8, 8))
    plt.scatter(actual_values, predictions, alpha=0.5)
    
    # Plot diagonal line (perfect prediction)
    min_val = min(min(actual_values), min(predictions))
    max_val = max(max(actual_values), max(predictions))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
    
    plt.xlabel('Actual Delay', fontsize=12)
    plt.ylabel('Predicted Delay', fontsize=12)
    plt.title('Prediction vs Actual', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.axis('equal')
    plt.tight_layout()
    plt.show()